# Hyperparameter Tuning & Model Selection

In [92]:

import numpy as np
import pandas as pd

np.random.seed(42)

n = 1500

df = pd.DataFrame({
    "equipment_age_months": np.random.randint(1, 121, n),
    "temperature_c": np.random.normal(45, 12, n).clip(15, 90),
    "vibration_mm_s": np.random.gamma(2, 1.5, n).clip(0, 15),
    "signal_strength_dbm": np.random.normal(-75, 12, n).clip(-120, -30),
    "packet_loss_pct": np.random.exponential(1.5, n).clip(0, 15),
    "traffic_load_pct": np.random.uniform(10, 100, n),
    "maintenance_days_ago": np.random.randint(1, 366, n),
    "equipment_type": np.random.choice(
        ["Router", "Switch", "Access Point", "Gateway"], n
    ),
    "region": np.random.choice(
        ["North", "South", "East", "West"], n
    ),
    "firmware_version": np.random.choice(
        ["v1.0", "v1.1", "v2.0"], n
    ),
    "weather_condition": np.random.choice(
        ["Clear", "Rainy", "Stormy", "Cloudy"], n
    )
})

# Generate the target using simulated fault-risk factors
risk_score = (
    -3.0
    + 0.018 * df["equipment_age_months"]
    + 0.035 * (df["temperature_c"] - 45)
    + 0.16 * df["vibration_mm_s"]
    + 0.22 * df["packet_loss_pct"]
    + 0.012 * df["traffic_load_pct"]
    + 0.002 * df["maintenance_days_ago"]
    + 0.65 * (df["weather_condition"] == "Stormy").astype(int)
    + 0.35 * (df["equipment_type"] == "Router").astype(int)
)

probability = 1 / (1 + np.exp(-risk_score))

df["network_fault"] = np.random.binomial(1, probability)

# Introduce missing values into selected features
for col in [
    "temperature_c",
    "signal_strength_dbm",
    "packet_loss_pct",
    "equipment_type",
    "firmware_version"
]:
    missing_idx = df.sample(frac=0.04, random_state=42).index
    df.loc[missing_idx, col] = np.nan

df.sample(8)


,equipment_age_months,temperature_c,vibration_mm_s,signal_strength_dbm,packet_loss_pct,traffic_load_pct,maintenance_days_ago,equipment_type,region,firmware_version,weather_condition,network_fault
820,116,54.752273,2.543245,-69.586941,1.296207,68.293817,103,Gateway,South,v1.1,Stormy,0
971,105,44.304193,6.724802,-86.355308,4.590394,36.685141,245,Router,West,v1.0,Cloudy,1
879,19,37.208813,1.789235,-88.277695,1.553060,32.660961,190,Switch,South,v2.0,Cloudy,0
1247,31,43.063869,1.845114,-83.716312,0.301954,72.771514,103,Access Point,East,v1.0,Cloudy,0
566,17,59.473841,0.314598,-64.461811,0.461898,45.783268,134,Gateway,East,v1.1,Rainy,0
615,69,53.416477,8.501494,-70.374323,0.459393,11.671365,21,Gateway,North,v1.1,Rainy,0
150,34,60.435223,5.021567,-77.067962,0.201673,28.403512,201,Gateway,East,v1.1,Stormy,1
547,5,49.762064,1.843973,-78.006724,0.030919,16.567150,270,Access Point,North,v1.0,Rainy,0


In [94]:
df.shape

(1500, 12)

In [96]:
df.duplicated().sum()

0

In [98]:
df.isnull().sum()

equipment_age_months     0
temperature_c           60
vibration_mm_s           0
signal_strength_dbm     60
packet_loss_pct         60
traffic_load_pct         0
maintenance_days_ago     0
equipment_type          60
region                   0
firmware_version        60
weather_condition        0
network_fault            0
dtype: int64

In [102]:
df.sample()

,equipment_age_months,temperature_c,vibration_mm_s,signal_strength_dbm,packet_loss_pct,traffic_load_pct,maintenance_days_ago,equipment_type,region,firmware_version,weather_condition,network_fault
209,45,37.525554,4.525053,-98.82796,0.873145,18.02859,177,Router,South,v1.1,Rainy,0


In [104]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   equipment_age_months  1500 non-null   int32  
 1   temperature_c         1440 non-null   float64
 2   vibration_mm_s        1500 non-null   float64
 3   signal_strength_dbm   1440 non-null   float64
 4   packet_loss_pct       1440 non-null   float64
 5   traffic_load_pct      1500 non-null   float64
 6   maintenance_days_ago  1500 non-null   int32  
 7   equipment_type        1440 non-null   object 
 8   region                1500 non-null   object 
 9   firmware_version      1440 non-null   object 
 10  weather_condition     1500 non-null   object 
 11  network_fault         1500 non-null   int32  
dtypes: float64(5), int32(3), object(4)
memory usage: 123.2+ KB


In [106]:
df.sample(8)

,equipment_age_months,temperature_c,vibration_mm_s,signal_strength_dbm,packet_loss_pct,traffic_load_pct,maintenance_days_ago,equipment_type,region,firmware_version,weather_condition,network_fault
1178,89,34.365356,0.907271,-67.936374,0.594508,68.188159,149,Access Point,North,v1.0,Cloudy,0
66,9,54.766367,3.378676,-88.656700,0.702461,21.894560,56,Access Point,South,v1.1,Rainy,1
826,36,72.604921,6.101266,-74.321038,2.112425,27.609967,330,Switch,West,v1.0,Rainy,1
433,4,NaN,2.776548,NaN,NaN,89.746657,194,NaN,South,NaN,Cloudy,0
1316,101,27.623324,4.896053,-72.661114,1.947315,42.470956,118,Switch,East,v1.0,Rainy,1
1124,78,53.811656,4.925520,-73.868648,0.877616,49.876834,270,Gateway,South,v2.0,Clear,1
895,87,48.139956,0.488467,-75.668807,0.482332,82.533534,283,Router,East,v2.0,Cloudy,1
1276,44,47.199603,6.304115,-89.058053,0.337822,81.423208,213,Router,East,v2.0,Clear,1


#### Q1. Inspect and prepare the dataset
- Inspect shape, data types, and descriptive statistics.
- Check missing values and duplicate rows.
- Examine the network_fault class distribution.
- Separate X and y.
- Identify numerical and categorical columns.
- Check whether any feature should be excluded and explain why.

In [109]:
df['network_fault'].value_counts(normalize=True)*100

network_fault
1    52.466667
0    47.533333
Name: proportion, dtype: float64

In [111]:
X=df.drop(columns='network_fault')
y=df['network_fault']

#### Q2. Create a stratified train/test split
- Use an 80/20 split with random_state=42.
- Preserve the class distribution using stratification.
- Print the shapes of X_train, X_test, y_train, and y_test.
- Compare target proportions across the full, training, and testing datasets.

In [114]:
from sklearn.model_selection import train_test_split

In [118]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,stratify=y,test_size=0.2)
num_cols=X_train.select_dtypes(include=['int','float']).columns.tolist()
cat_cols=X_train.select_dtypes(include='object').columns.tolist()

#### Q3. Build the preprocessing pipeline
Create separate preprocessing pipelines:
- Numerical features: median imputation and StandardScaler.
- Categorical features: most-frequent imputation and OneHotEncoder(handle_unknown="ignore").
- Combine them using ColumnTransformer.
- Integrate preprocessing with LogisticRegression(max_iter=1000) using Pipeline.
- Fit the complete pipeline on the training data only.

In [148]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.metrics import(
precision_score,
recall_score,
accuracy_score,
f1_score,
roc_auc_score,
confusion_matrix
)

In [127]:
num_pipeline=Pipeline([
    ('impute',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
cat_pipeline=Pipeline([
    ('impute',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])
preprocessor=ColumnTransformer([
    ('Numeric_Columns',num_pipeline,num_cols),
    ('Categorical_Columns',cat_pipeline,cat_cols)
])

model_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('classifier',LogisticRegression(max_iter=1000))
])

model_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('Numeric_Columns',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['equipment_age_months',
                                                   'temperature_c',
                                                   'vibration_mm_s',
                                                   'signal_strength_dbm',
                                                   'packet_loss_pct',
                                                   'traffic_load_pct',
                                                   'maintenance_days_ago']),
                                                 ('Categorical_Columns',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['equipment_type', 'region',
                                                   'firmware_version',
                                                   'weather_condition'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

#### Q4. Establish a baseline model
- Train the baseline Logistic Regression pipeline.
- Generate predictions on the test set.
- Calculate accuracy, precision, recall, F1-score, and ROC-AUC.
- Print the confusion matrix.
- Explain why accuracy alone may be misleading if network faults are uncommon.

In [152]:
y_pred=model_pipeline.predict(X_test)
y_proba=model_pipeline.predict_proba(X_test)[:,1]

print('Accuracy:',accuracy_score(y_test,y_pred))
print('Precision:',precision_score(y_test,y_pred))
print('Recall:',recall_score(y_test,y_pred))
print('F1:',f1_score(y_test,y_pred))
print('ROC_AUC:',roc_auc_score(y_test,y_proba))

print('Confustion Matrix\n',confusion_matrix(y_test,y_pred))

Accuracy: 0.6733333333333333
Precision: 0.6832298136645962
Recall: 0.7006369426751592
F1: 0.6918238993710691
ROC_AUC: 0.7251792793194067
Confustion Matrix
 [[ 92  51]
 [ 47 110]]


#### Model Evaluation Report

The model achieved an accuracy of 67.33% with a precision of 68.32%, recall of 70.06%, and F1-score of 69.18%. The ROC-AUC score of 0.7252 indicates a reasonable ability to distinguish between the two classes.

The confusion matrix shows that the model correctly predicted 92 negative cases and 110 positive cases, while misclassifying 51 negative cases as positive and 47 positive cases as negative.

Overall, the model demonstrates moderate classification performance, with reasonably balanced precision and recall. Further improvement through hyperparameter tuning, feature engineering, and class-imbalance handling may enhance its predictive performance.

#### Q5. Tune Logistic Regression using GridSearchCV
Tune the following parameters:
- classifier__C: [0.01, 0.1, 1, 10]
- classifier__class_weight: [None, "balanced"]
Requirements:
- Use StratifiedKFold with 5 splits, shuffling enabled, and random_state=42.
- Use ROC-AUC as the scoring metric.
- Set refit=True.
- Use n_jobs=-1.
- Display the best parameters and best cross-validation ROC-AUC.
- Store the best estimator for later comparison.

In [156]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold


In [162]:
param_grid=[{
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__class_weight': [None, "balanced"]
}]

skf=StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search=GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    n_jobs=-1,
    scoring='roc_auc',
    refit=True,
    cv=skf
)

grid_search.fit(X_train,y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('Numeric_Columns',
                                                                         Pipeline(steps=[('impute',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['equipment_age_months',
                                                                          'temperature_c',
                                                                          'vibration_mm_s',
                                                                          'signal_strength_dbm',
                                                                          'packet_loss_pc...
                                                                         Pipeline(steps=[('impute',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['equipment_type',
                                                                          'region',
                                                                          'firmware_version',
                                                                          'weather_condition'])])),
                                       ('classifier',
                                        LogisticRegression(max_iter=1000))]),
             n_jobs=-1,
             param_grid=[{'classifier__C': [0.01, 0.1, 1, 10],
                          'classifier__class_weight': [None, 'balanced']}],
             scoring='roc_auc')

In [174]:
print('grid search: BEST PARAMETERS',grid_search.best_params_)
print('grid search: BEST CV SCORES',grid_search.best_score_)

best_logistic_model = grid_search.best_estimator_

grid search: BEST PARAMETERS {'classifier__C': 0.01, 'classifier__class_weight': None}
grid search: BEST CV SCORES 0.7420356446672236


#### Q6. Tune a Decision Tree
Create a new pipeline using the same preprocessing and a DecisionTreeClassifier.

Tune:
- classifier__max_depth: [3, 5, 8, None]
- classifier__min_samples_leaf: [1, 5, 10]
- classifier__class_weight: [None, "balanced"]
Use the same five-fold stratified cross-validation and ROC-AUC scoring.

Report the best parameters and best cross-validation ROC-AUC.

In [195]:
from sklearn.tree import DecisionTreeClassifier

param_grid_tree={'classifier__max_depth': [3, 5, 8, None],
'classifier__min_samples_leaf': [1, 5, 10],
'classifier__class_weight': [None, "balanced"]
}

In [197]:
tree_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('classifier',DecisionTreeClassifier(random_state=42))
])

tree_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('Numeric_Columns',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['equipment_age_months',
                                                   'temperature_c',
                                                   'vibration_mm_s',
                                                   'signal_strength_dbm',
                                                   'packet_loss_pct',
                                                   'traffic_load_pct',
                                                   'maintenance_days_ago']),
                                                 ('Categorical_Columns',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['equipment_type', 'region',
                                                   'firmware_version',
                                                   'weather_condition'])])),
                ('classifier', DecisionTreeClassifier(random_state=42))])

In [201]:
tree_grid_search = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=param_grid_tree,
    scoring='roc_auc',
    cv=skf,
    refit=True,
    n_jobs=-1
)
tree_grid_search.fit(X_train,y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('Numeric_Columns',
                                                                         Pipeline(steps=[('impute',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['equipment_age_months',
                                                                          'temperature_c',
                                                                          'vibration_mm_s',
                                                                          'signal_strength_dbm',
                                                                          'packet_loss_pc...
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['equipment_type',
                                                                          'region',
                                                                          'firmware_version',
                                                                          'weather_condition'])])),
                                       ('classifier',
                                        DecisionTreeClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__class_weight': [None, 'balanced'],
                         'classifier__max_depth': [3, 5, 8, None],
                         'classifier__min_samples_leaf': [1, 5, 10]},
             scoring='roc_auc')

In [203]:
print("Best Parameters:", tree_grid_search.best_params_)
print("Best Cross-Validation ROC-AUC:", tree_grid_search.best_score_)

# Store the best fitted model
best_tree_model = tree_grid_search.best_estimator_

Best Parameters: {'classifier__class_weight': None, 'classifier__max_depth': 3, 'classifier__min_samples_leaf': 10}
Best Cross-Validation ROC-AUC: 0.6566903369534949
